# Data Collection
Download SEC bulk files, pull price data, merge, and filter sample.

In [1]:
import sys
import os
import pandas as pd
import yfinance as yf
import numpy as np

# Add src to path
sys.path.append('../src')
import data_utils

# Parameters
years = [2022, 2023]
quarters = [1, 2, 3, 4]
target_tickers = ['AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'TSLA', 'BRK-B', 'JNJ', 'JPM', 'V',
                  'PG', 'UNH', 'HD', 'MA', 'DIS', 'NVDA', 'PYPL', 'BAC', 'VZ', 'ADBE']

# Download SEC Data
all_sec_data = []
for year in years:
    for quarter in quarters:
        print(f"Downloading {year} Q{quarter}...")
        zip_path = data_utils.download_sec_form4_data(year, quarter, '../data/raw')
        if zip_path:
            df_raw = data_utils.extract_and_load_nonderiv(zip_path)
            df_processed = data_utils.process_sec_data(df_raw, target_tickers)
            all_sec_data.append(df_processed)

if all_sec_data:
    df_sec = pd.concat(all_sec_data, ignore_index=True)
    df_sec.to_csv('../data/processed/sec_insider_trades.csv', index=False)
    print(f"Saved processed SEC data. Shape: {df_sec.shape}")


File 2022q1_form4.zip already exists. Skipping download.
File 2022q2_form4.zip already exists. Skipping download.
File 2022q3_form4.zip already exists. Skipping download.
File 2022q4_form4.zip already exists. Skipping download.
File 2023q1_form4.zip already exists. Skipping download.
File 2023q2_form4.zip already exists. Skipping download.
File 2023q3_form4.zip already exists. Skipping download.
File 2023q4_form4.zip already exists. Skipping download.
Saved processed SEC data. Shape: (3041, 8)


In [2]:
# Download Stock Prices
start_date = '2021-01-01' # Buffer for estimation window
end_date = '2024-03-31'

prices = data_utils.download_stock_prices(target_tickers, start_date, end_date)
prices.to_csv('../data/processed/stock_prices.csv')

benchmark = data_utils.download_market_benchmark(start_date, end_date)
benchmark.to_csv('../data/processed/market_benchmark.csv')
print("Saved price data.")

# Fetch Market Caps
print("Fetching Market Caps...")
market_caps = {}
for ticker in target_tickers:
    try:
        t = yf.Ticker(ticker)
        mc = t.info.get('marketCap', np.nan)
        market_caps[ticker] = mc
    except Exception as e:
        market_caps[ticker] = np.nan
        
mc_df = pd.DataFrame(list(market_caps.items()), columns=['Ticker', 'MarketCap'])
mc_df.to_csv('../data/processed/market_caps.csv', index=False)
print("Saved market caps.")


[*********************100%***********************]  20 of 20 completed
[*********************100%***********************]  1 of 1 completed


Saved price data.
Fetching Market Caps...
Saved market caps.
